# Searching

This notebook builds search methods from simple to efficient, then benchmarks them.

**Learning Goals**
- Implement linear and binary search correctly
- Use hash-based lookup for repeated membership checks
- Compare runtime trends with repeated timing
- Choose a search strategy from data constraints

In [1]:
from time import perf_counter
import random

## Linear Search

Linear search checks each element from left to right until a match is found.

- Time complexity: `O(n)`
- Works on unsorted data
- Best when data is small or only one lookup is needed

In [2]:
def linear_search(items, target):
    """Return index of target in items, or -1 if not found."""
    for i, value in enumerate(items):
        if value == target:
            return i
    return -1

sample = [7, 4, 9, 1, 3]
print(linear_search(sample, 9))   # 2
print(linear_search(sample, 8))   # -1

2
-1


In [2]:
### Exercise
# Write `min_value(nums)` using the same input-process-output idea.
# Requirements:
# 1. Return `None` for an empty list.
# 2. Return the smallest value otherwise.
### Your code starts here.


### Your code ends here.

In [3]:
### Solution

def min_value(nums):
    if not nums:
        return None

    current_min = nums[0]
    for x in nums[1:]:
        if x < current_min:
            current_min = x
    return current_min

> **Next:** If data can be kept sorted and repeated lookups are needed, binary search is usually the better choice.

## Binary Search (Iterative)

Binary search repeatedly halves the candidate range.

- Time complexity: `O(log n)`
- Requires sorted input

In [3]:
def binary_search(sorted_items, target):
    """Return index of target in sorted_items, or -1 if not found.

    Note: the returned index is a position in *sorted_items*, not the original
    unsorted list. Use linear_search when you need the original-list position.
    """
    left, right = 0, len(sorted_items) - 1

    while left <= right:
        mid = (left + right) // 2
        guess = sorted_items[mid]

        if guess == target:
            return mid
        if guess < target:
            left = mid + 1
        else:
            right = mid - 1

    return -1

sorted_sample = [1, 3, 4, 7, 9, 14, 20]
print(binary_search(sorted_sample, 14))  # 5
print(binary_search(sorted_sample, 2))   # -1

5
-1


## Hash-Based Lookup

When the task is membership testing, a `set`/`dict` is often the most practical choice.
Average lookup is `O(1)` and independent of order.

Connection to Chapter 08:
- This is the same list-vs-dict/set lookup idea you measured earlier.
- Here, we place that `O(1)` option next to linear (`O(n)`) and binary (`O(log n)`) search.

In [4]:
numbers = [random.randint(1, 20_000) for _ in range(15_000)]
lookup_set = set(numbers)

target = numbers[len(numbers) // 2]
missing = 999_999

print('target in list:', target in numbers)
print('target in set :', target in lookup_set)
print('missing in list:', missing in numbers)
print('missing in set :', missing in lookup_set)

target in list: True
target in set : True
missing in list: False
missing in set : False


## Correctness Checks

Before benchmarking, verify edge cases and duplicate behavior.

In [5]:
tests = [
    ([], 5),
    ([10], 10),
    ([10], 5),
    ([1, 2, 2, 2, 3], 2),
    (list(range(100)), 99),
]

for arr, target in tests:
    lin_idx = linear_search(arr, target)
    arr_sorted = sorted(arr)
    bin_idx = binary_search(arr_sorted, target)

    # verify hash-based lookup agrees with membership result
    assert (target in set(arr)) == (target in arr)

    if target in arr:
        assert lin_idx != -1
        assert arr[lin_idx] == target
        assert bin_idx != -1
        assert arr_sorted[bin_idx] == target
    else:
        assert lin_idx == -1
        assert bin_idx == -1

print('All checks passed.')

All checks passed.


## Timing Comparison

Benchmark design here:
- same target pattern across methods
- repeated runs per input size
- compare trends, not tiny decimal differences

Timing continuity:
- If you used `time.time()` in earlier chapters, that is still valid for rough timing.
- We use `time.perf_counter()` here for higher-resolution benchmarks.

In [6]:
def average_runtime(fn, *args, repeats=30):
    total = 0.0
    for _ in range(repeats):
        start = perf_counter()
        fn(*args)
        total += perf_counter() - start
    return total / repeats

sizes = [10_000, 50_000, 100_000, 200_000]
print(f"{'n':>10} {'linear (s)':>15} {'binary (s)':>15} {'set in (s)':>15}")

for n in sizes:
    data = list(range(n))
    target = n - 1
    data_set = set(data)

    t_linear = average_runtime(linear_search, data, target)
    t_binary = average_runtime(binary_search, data, target)
    t_set = average_runtime(lambda s, x: x in s, data_set, target)

    print(f"{n:>10} {t_linear:>15.8f} {t_binary:>15.8f} {t_set:>15.8f}")

         n      linear (s)      binary (s)      set in (s)
     10000      0.00021811      0.00000129      0.00000008
     50000      0.00090510      0.00000111      0.00000008
    100000      0.00158124      0.00000115      0.00000019


    200000      0.00314533      0.00000116      0.00000012


In [5]:
### Exercise
# Modify `classify_scores` to include grade `D` for scores below 70.
### Your code starts here.


### Your code ends here.

In [6]:
### Solution

def classify_scores_with_d(scores):
    labels = []
    for s in scores:
        if s >= 90:
            labels.append('A')
        elif s >= 80:
            labels.append('B')
        elif s >= 70:
            labels.append('C')
        else:
            labels.append('D')
    return labels

In [8]:
### Exercise
# In one or two sentences, explain why `O(n log n)` scales better than `O(n^2)` for large `n`.
### Your code starts here.


### Your code ends here.

In [9]:
### Solution

# Example answer:
# `O(n log n)` grows much more slowly than `O(n^2)` as n increases.
# For large inputs, this difference leads to substantially lower runtime.

In [7]:
### Exercise
# Implement `binary_search_first(sorted_items, target)`.
# Requirements:
# 1. Return the first index of `target` when duplicates exist.
# 2. Return `-1` when the target is absent.
# 3. Validate with at least: empty input, all duplicates, target missing.
### Your code starts here.


### Your code ends here.

In [8]:
### Solution

def binary_search_first(sorted_items, target):
    left, right = 0, len(sorted_items) - 1
    result = -1

    while left <= right:
        mid = (left + right) // 2
        guess = sorted_items[mid]

        if guess == target:
            result = mid
            right = mid - 1
        elif guess < target:
            left = mid + 1
        else:
            right = mid - 1

    return result

dup_data = [1, 2, 2, 2, 4, 4, 5]
print(binary_search_first(dup_data, 2))  # expected: 1
print(binary_search_first(dup_data, 4))  # expected: 4
print(binary_search_first(dup_data, 3))  # expected: -1

1
4
-1
